re-doing my setup for the ANTARES filter system since the pitt-google setup is having issues. i'm testing a filter using the [structure of a filter](https://nsf-noirlab.gitlab.io/csdc/antares/devkit/learn/structure-of-a-filter/) example, the [anomaly transient dmdt filter](https://nsf-noirlab.gitlab.io/csdc/antares/devkit/reference/filters/malanchev_anomaly_transient_dmdt/), and the [editing real loci example](https://nsf-noirlab.gitlab.io/csdc/antares/devkit/learn/testing-filters/#building-test-cases) as a baseline for it

In [9]:
%pip install antares_client

  Using cached antares_client-1.14.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached bson-0.5.10.tar.gz (10 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached click-8.3.3-py3-none-any.whl.metadata (2.6 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached marshmallow_jsonapi-0.24.0-py2.py3-none-any.whl.metadata (5.2 kB)
  Using cached astropy_healpix-1.1.3-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
Using cached antares_client-1.14.0-py3-none-any.whl (19 kB)
Using cached marshmallow-3.26.2-py3-none-any.whl (50 kB)
Using cached marshmallow_jsonapi-0.24.0-py2.py3-none-any.whl (14 kB)
Using cached astropy_healpix-1.1.3-cp310-abi3-macosx_11_0_arm64.whl (82 kB)
Using cached click-8.3.3-py3-none-any.whl (110 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for bson (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [99

In [1]:
from antares_devkit.models import BaseFilter
from antares_devkit.models import DevKitLocus
from antares_client import search

ModuleNotFoundError: No module named 'antares_devkit'

In [25]:
client_locus = search.get_by_id("ANT2023q6hwmp481mvb") # 2
locus_dict = client_locus.to_devkit()
locus_dict

{'id': 'ANT2023q6hwmp481mvb',
 'ra': 187.43561054509746,
 'dec': 7.877639089460132,
 'user_tags': ['refitt_newsources_snrcut',
  'lc_feature_extractor',
  'in_shadow_virgo',
  'high_amplitude_transient_candidate'],
 'old_properties': {'num_alerts': 382,
  'num_mag_values': 190,
  'brightest_alert_id': 'ztf_candidate:3051258570615015004',
  'brightest_alert_magnitude': 18.120290756225586,
  'brightest_alert_observation_time': 60805.25857640011,
  'newest_alert_id': 'ztf_candidate:3360413494915015026',
  'newest_alert_magnitude': 18.66741371154785,
  'newest_alert_observation_time': 61114.41349539999,
  'oldest_alert_id': 'ztf_candidate:2215381030615015011',
  'oldest_alert_magnitude': 19.706274032592773,
  'oldest_alert_observation_time': 59969.38103009993,
  'is_corrected': 'true',
  'ztf_object_id': 'ZTF24aanbbes',
  'ztf_ssnamenr': 'null',
  'survey': {'ztf': {'id': ['ZTF22aaiczth', 'ZTF24aanbbes'],
    'rcid': [6, 49],
    'field': [473, 525],
    'ssnamenr': ['null']},
   'lsst': {

In [3]:
locus1 = DevKitLocus.model_validate(locus_dict)

In [2]:
# directly from the Anomalytransientdmdt antares filter
def get_detections(photometry, band):
    """Extract clean light curve in given band from locus photometry"""
    from astropy.table import MaskedColumn

    band_lc = photometry[photometry["ant_passband"] == band]
    idx = ~MaskedColumn(band_lc["ant_mag"]).mask
    detections = remove_simultaneous_alerts(band_lc[idx])
    return detections

def remove_simultaneous_alerts(table):
    """Remove alert duplicates"""
    import numpy as np

    dt = np.diff(table["ant_mjd"], append=np.inf)
    return table[dt != 0]

In [3]:
from astropy.modeling import models, fitting

fit = fitting.LinearLSQFitter()
line_init = models.Linear1D()

# make a function that determines if the object is highly variable
# building with the idea that the most recent observation was last night
slope_change_threshold = 1 # mag /mjd
time_pass_threshold = 20 # days
def get_ext_variables(photometry):
    # what i want to do:
    i = 1
    mjd_now = photometry[0]["ant_mjd"]
    if (mjd_now - photometry[i]["ant_mjd"]) < time_pass_threshold:
        fitted_line = fit(line_init, photometry[0:i]["ant_mag"], photometry[0:i]["ant_mjd"])
        slope = fitted_line.slope
        if abs(slope) < slope_change_threshold:
            # no High variability
            i +=1
        else:
            return photometry
    elif (mjd_now - photometry[i]["ant_mjd"]) > time_pass_threshold:
        # no Recent variability
        return 0

In [4]:
class HighMagChange(BaseFilter):
    """
    This filter finds locus with recent extreme brightness variability.
    The thresholds used are r_slope > # within # days.

    Current assumptions:
    - real time alert ingestion
    - slope
    - number of days for this slope detection
    - band (r)
    """

    SLACK_CHANNEL = "#fliter-extreme-recent-variables"
    
    TRIGGERING_SURVEY = "lsst"
    REQUIRED_ALERT_PROPERTIES = [
        "lsst_diaSource_diaObjectId",
        # "lsst_diaSource_band",
        # "lsst_diaSource_midpointMjdTai",
        # "lsst_diaSource_scienceFlux", 
        # "lsst_diaSource_scienceFluxErr",
        # "lsst_diaSource_templateFlux", 
        # "lsst_diaSource_templateFluxErr",
        # "lsst_diaObject_i_psfFluxMaxSlope",
        "ant_mjd",
        "ant_mag",
        "ant_maglim",
        "ant_magerr",
        "ant_passband"
    ]
    OUTPUT_LOCUS_TAGS = [
        {
            "name": "extreme_transient_candidate_v1",
            "description": "Locus with alert(s) having high change in mag.",
        },
    ]

    band = "r"
    min_obs_count = 4
    faint_mag_threshold = 30

    # def setup(self):
        # idk if i need this bit,,,,,,,,,
    
    def _run(self, locus):
        import numpy as np
        from astropy.time import Time
        
        # get clean lightcurve in one band, with the TimeSeries being in reverse order (so newest is first)
        # for now, im focusing on the r band
        lightcurve = locus.lightcurve(self.TRIGGERING_SURVEY)
        detections = get_detections(lightcurve, self.band)
        detections.sort("ant_mjd", reverse=True)
        # print(detections[0]["ant_mag"])

        if detections[0]["ant_mag"] > self.faint_mag_threshold:
            # too dim to collect spectra
            return
        
        if len(detections) < self.min_obs_count:
            # not enough data
            return

        # det if the lightcurve shows high enough variability
        variables = get_ext_variables(detections)

        if variables == 0:
            # no recent, high variability seen in the lightcurve
            # print("no recent, high variability seen in the lightcurve")
            return "no recent, high variability seen in the lightcurve"
        else:
            return variables # this is temporary
            # locus.tag("extreme_recent_variable")
            
        # The threshold is dependent on the band that is being imaged.
        # These thresholds should flag ~2-3% of alerts.
        # snr_threshold = {
        #     1: 50.0,
        #     2: 55.0,
        # }  # Only have thresholds for passbands ztf_fid 1 and 2, i.e., g and R passbands.

        # Determine the passband of the most recent alert at this locus.
        # fid = locus.alert.properties["ztf_fid"]  # current alert passband
        # if fid not in snr_threshold.keys():
        #     # print(f"ZTF passband id {fid} is not supported by this filter.")
        #     return  # Do nothing.

        # # Calculate the SNR of the latest alert
        # sigmapsf = locus.alert.properties["ztf_sigmapsf"]  # current alert sigma
        # threshold = snr_threshold[fid]
        # alert_snr = 1.0 / sigmapsf

        # alert_id = locus.alert.alert_id  # current alert_id
        # ztf_object_id = locus.properties['ztf_object_id']  # ZTF Object ID

        # Tag this locus if the latest alert meets our SNR criteria
        # if alert_snr > threshold:
        #     # print('High SNR detected')
        #     locus.tag("high_snr")

In [109]:
here = HighMagChange()
here.run(locus1)

Attempt to correct magnitudes failed, missing fields
Attempt to correct magnitudes failed, missing fields
        alert_id             ant_mjd       ... ant_magllim_corrected
----------------------- ------------------ ... ---------------------
lsst:170063717281562805  61098.24137255241 ...                    --
lsst:170063717147345055 61098.240903781785 ...                    --
lsst:170063716878909646  61098.23997026903 ...                    --
lsst:170063716744691860  61098.23950327612 ...                    --
lsst:170063716460527738 61098.238440843794 ...                    --
lsst:170063716325785724  61098.23797378523 ...                    --
lsst:170063716102963357 61098.236850402696 ...                    --
lsst:170063715969269886  61098.23637889169 ...                    --
lsst:170063715832955036  61098.23591076338 ...                    --
lsst:170063715699785896  61098.23544140398 ...                    --
                    ...                ... ...                   .

FilterReturn(status='Succeeded', exception_name=None, exception_message=None, traceback=None)

ques:
1. how does this work if youre trying to look at multiple loci? does each loci run thru the filter individually, or are they sent in as a packet?

In [7]:
loci = ["ANT2020h3zzw", "ANT2024lrwe5t0dhfyk", "ANT2020mbdmm", "ANT2020gnbyk", "ANT2020gjbf6"]
for locus in loci:
    client_locus = search.get_by_id(locus)
    locus_dict = client_locus.to_devkit()
    valid_locus = DevKitLocus.model_validate(locus_dict)
    here = HighMagChange()
    result = here.run(valid_locus)
    print(result)

status='Skipped' exception_name=None exception_message=None traceback=None
status='Skipped' exception_name=None exception_message=None traceback=None
status='Skipped' exception_name=None exception_message=None traceback=None
status='Skipped' exception_name=None exception_message=None traceback=None
status='Skipped' exception_name=None exception_message=None traceback=None


next step: using more data to test if my filter would work against an active alert stream. use the [getting data](https://nsf-noirlab.gitlab.io/csdc/antares/devkit/learn/getting-data/) page to figure it out

In [ ]:
from antares_client.search import search as ant_search
fro